In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
pasta_trabalho = '/content/drive/MyDrive/Trabalho_BGP'
os.makedirs(pasta_trabalho, exist_ok=True)
print(f"Pasta configurada em: {pasta_trabalho}")

Mounted at /content/drive
Pasta configurada em: /content/drive/MyDrive/Trabalho_BGP


In [ ]:
!wget -P /content/drive/MyDrive/Trabalho_BGP https://data.ris.ripe.net/rrc00/2026.07/bview.20260701.0000.gz

--2026-07-11 17:33:18--  https://data.ris.ripe.net/rrc00/2026.07/bview.20260701.0000.gz
Resolving data.ris.ripe.net (data.ris.ripe.net)... 193.0.11.24, 2001:67c:2e8:25::c100:b18
Connecting to data.ris.ripe.net (data.ris.ripe.net)|193.0.11.24|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 415445025 (396M) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/Trabalho_BGP/bview.20260701.0000.gz’

bview.20260701.0000 100%[===================>] 396.20M  24.8MB/s    in 14s     

2026-07-11 17:33:33 (27.6 MB/s) - ‘/content/drive/MyDrive/Trabalho_BGP/bview.20260701.0000.gz’ saved [415445025/415445025]



In [ ]:
import requests
import sys

def buscar_asns_taiwan():
    url = "https://stat.ripe.net/data/country-resource-list/data.json?resource=TW"
    print("Consultando a API RIPE stat para buscar a lista de ASNs de Taiwan (TW)...")

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        dados = response.json()

        asns = []

        # Acessando o bloco de dados onde a lista de ASNs fica guardada
        recursos_asn = dados.get('data', {}).get('resources', {}).get('asn', [])

        for item in recursos_asn:
            # A API pode retornar um intervalo (ex: "456-460") ou um número único (ex: "123")
            item = str(item).strip()
            if '-' in item:
                inicio, fim = item.split('-')
                # Adiciona todos os ASNs do intervalo à nossa lista
                asns.extend(range(int(inicio), int(fim) + 1))
            else:
                asns.append(int(item))

        if not asns:
            print("\nNenhum ASN encontrado.")
            print(dados)
            return

        # Remove duplicatas, ordena e salva no drive
        asns_unicos = sorted(list(set(asns)))
        caminho_arquivo = "/content/drive/MyDrive/Trabalho_BGP/asns_taiwan.txt"

        with open(caminho_arquivo, "w", encoding="utf-8") as arquivo:
            for asn in asns_unicos:
                arquivo.write(f"{asn}\n")

        print(f"Sucesso! {len(asns_unicos)} ASNs foram salvos com sucesso no arquivo '{caminho_arquivo}'.")

    except Exception as e:
        print(f"Erro ao executar a consulta: {e}", file=sys.stderr)

buscar_asns_taiwan()

Consultando a API RIPE stat para buscar a lista de ASNs de Taiwan (TW)...
Sucesso! 468 ASNs foram salvos com sucesso no arquivo '/content/drive/MyDrive/Trabalho_BGP/asns_taiwan.txt'.


In [ ]:
!pip install mrtparse

In [ ]:
import sys
import csv
import gzip
import traceback
from collections import defaultdict
from mrtparse import Reader

ARQUIVO_ASNS = '/content/drive/MyDrive/Trabalho_BGP/asns_taiwan.txt'
ARQUIVO_RIB = '/content/drive/MyDrive/Trabalho_BGP/bview.20260701.0000.gz'
ARQUIVO_SAIDA = '/content/drive/MyDrive/Trabalho_BGP/upstreams_taiwan.csv'

def extrair_upstreams():
    print("Iniciando o processamento da tabela BGP...")

    taiwan_asns = set()
    try:
        with open(ARQUIVO_ASNS, 'r', encoding='utf-8') as f:
            for linha in f:
                asn = linha.strip()
                if asn.isdigit(): taiwan_asns.add(asn)
    except FileNotFoundError:
        print("Erro: Arquivo asns_taiwan.txt não encontrado.")
        return

    relacoes_transito = defaultdict(int)
    entradas_processadas = 0

    print("Descompactando e lendo o arquivo RIB.")
    with gzip.open(ARQUIVO_RIB, 'rb') as f_gz:
        try:
            for m in Reader(f_gz):
                mrt_data = m.data

                # Extraindo a chave do dicionário
                chaves_tipo = list(mrt_data.get('type', {}).keys())
                if not chaves_tipo or chaves_tipo[0] != 13:
                    continue

                entradas_processadas += 1
                if entradas_processadas % 100000 == 0:
                    print(f"Processando... {entradas_processadas} mensagens lidas.")

                for entrada in mrt_data.get('rib_entries', []):
                    as_path_bruto = []

                    for attr in entrada.get('path_attributes', []):
                        chaves_attr = list(attr.get('type', {}).keys())
                        if chaves_attr and chaves_attr[0] == 2: # AS_PATH
                            for seg in attr.get('value', []):
                                chaves_seg = list(seg.get('type', {}).keys())
                                if chaves_seg and chaves_seg[0] == 2: # AS_SEQUENCE
                                    as_path_bruto.extend(seg.get('value', []))

                    if len(as_path_bruto) < 2:
                        continue

                    # Colapsar AS Prepending
                    caminho = [str(as_path_bruto[0])]
                    for asn in as_path_bruto[1:]:
                        if str(asn) != caminho[-1]:
                            caminho.append(str(asn))

                    if len(caminho) >= 2 and caminho[-1] in taiwan_asns:
                        # Adicionando a aresta (Destino, Upstream)
                        relacoes_transito[(caminho[-1], caminho[-2])] += 1

        except Exception as e:
            print(f"\nErro na iteração: {type(e).__name__}: {e}")
            traceback.print_exc()
            return

    with open(ARQUIVO_SAIDA, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['asn_destino', 'asn_upstream', 'frequencia'])
        for (dest, up), freq in sorted(relacoes_transito.items(), key=lambda x: x[1], reverse=True):
            writer.writerow([dest, up, freq])

    print(f"\nSucesso! {len(relacoes_transito)} relações de upstream consolidadas.")
    print(f"Dados salvos em {ARQUIVO_SAIDA}")

extrair_upstreams()

Iniciando o processamento da tabela BGP...
Descompactando e lendo o arquivo RIB. Isso pode levar alguns minutos...
Processando... 100000 mensagens lidas.
Processando... 200000 mensagens lidas.
Processando... 300000 mensagens lidas.
Processando... 400000 mensagens lidas.
Processando... 500000 mensagens lidas.
Processando... 600000 mensagens lidas.
Processando... 700000 mensagens lidas.
Processando... 800000 mensagens lidas.
Processando... 900000 mensagens lidas.
Processando... 1000000 mensagens lidas.
Processando... 1100000 mensagens lidas.
Processando... 1200000 mensagens lidas.
Processando... 1300000 mensagens lidas.

Sucesso! 981 relações de upstream consolidadas.
Dados salvos em /content/drive/MyDrive/Trabalho_BGP/upstreams_taiwan.csv


In [ ]:
import pandas as pd
import requests
import time
import sys

ARQUIVO_ENTRADA = '/content/drive/MyDrive/Trabalho_BGP/upstreams_taiwan.csv'
ARQUIVO_SAIDA = '/content/drive/MyDrive/Trabalho_BGP/upstreams_enriquecidos.csv'
ARQUIVO_UPSTREAMS = '/content/drive/MyDrive/Trabalho_BGP/top_upstreams_absolutos.csv'
ARQUIVO_DESTINOS = '/content/drive/MyDrive/Trabalho_BGP/top_destinos_absolutos.csv'

def enriquecer_e_analisar():
    print("Carregando os dados brutos...")
    try:
        df = pd.read_csv(ARQUIVO_ENTRADA)
    except FileNotFoundError:
        print(f"Erro: Arquivo '{ARQUIVO_ENTRADA}' não encontrado.")
        return

    asns_unicos = df['asn_upstream'].unique()
    print(f"Encontrados {len(asns_unicos)} ASNs provedores únicos. Iniciando consultas à API...")

    # Dicionário de cache em memória para evitar consultas repetidas
    cache_paises = {}

    # Consultando a RIPE stat Data API (Endpoint: whois)
    for i, asn in enumerate(asns_unicos, 1):
        if asn not in cache_paises:
            # O endpoint whois é o mais confiável para extrair o Country Code
            url = f"https://stat.ripe.net/data/whois/data.json?resource=AS{asn}"
            pais = "Desconhecido"

            try:
                resposta = requests.get(url, timeout=5)
                if resposta.status_code == 200:
                    dados = resposta.json()
                    # Navegando pelos registros para encontrar a chave 'country'
                    for record in dados.get('data', {}).get('records', []):
                        for item in record:
                            if item.get('key') == 'country':
                                pais = item.get('value').upper()
                                break
                        if pais != "Desconhecido":
                            break
            except Exception as e:
                # Se a rede falhar, ele mantém como Desconhecido e segue em frente
                pass

            cache_paises[asn] = pais
            time.sleep(0.1) # Atraso de 100ms para não ser bloqueado pela API

        if i % 20 == 0:
            print(f"-> Consultados {i}/{len(asns_unicos)} ASNs...")

    print("Consultas concluídas! Mapeando os países no DataFrame...")

    # Criando a nova coluna de países
    df['pais_upstream'] = df['asn_upstream'].map(cache_paises)

    # Agrupando upstreams
    df_upstreams = df.groupby(['asn_upstream', 'pais_upstream'])['frequencia'].sum().reset_index()
    df_upstreams = df_upstreams.sort_values(by='frequencia', ascending=False)

    # Agrupando destinos
    df_destinos = df.groupby('asn_destino')['frequencia'].sum().reset_index()
    df_destinos = df_destinos.sort_values(by='frequencia', ascending=False)

    # Agrupando por Países
    df_paises = df.groupby('pais_upstream')['frequencia'].sum().reset_index()
    df_paises = df_paises.sort_values(by='frequencia', ascending=False)

    # Salvando os arquivos
    df.to_csv(ARQUIVO_SAIDA, index=False)
    df_upstreams.to_csv(ARQUIVO_UPSTREAMS, index=False)
    df_destinos.to_csv(ARQUIVO_DESTINOS, index=False)
    print("\nArquivos CSV salvos com sucesso no Google Drive!")

    # Cálculo e exibição das métricas
    print("\n" + "="*40)
    print("MÉTRICAS QUANTITATIVAS DO PROJETO (BGP)")
    print("="*40)

    print("\n[A] TOP 5 ASes PROVEDORES (UPSTREAMS ABSOLUTOS):")
    print(df_upstreams.head(5).to_string(index=False))

    print("\n[B] TOP 5 PAÍSES QUE MAIS FORNECEM ROTAS:")
    print(df_paises.head(5).to_string(index=False))

    # Cálculo da fração estrangeira vs doméstica
    total_rotas = df['frequencia'].sum()
    rotas_domesticas = df[df['pais_upstream'] == 'TW']['frequencia'].sum()
    rotas_estrangeiras = total_rotas - rotas_domesticas

    perc_estrangeiro = (rotas_estrangeiras / total_rotas) * 100
    perc_domestico = (rotas_domesticas / total_rotas) * 100

    print("\n[C] DEPENDÊNCIA DE TRÂNSITO ESTRANGEIRO:")
    print(f"-> Tráfego dependente de infraestrutura internacional: {perc_estrangeiro:.2f}%")
    print(f"-> Tráfego provido por infraestrutura doméstica (TW): {perc_domestico:.2f}%")
    print("="*40 + "\n")

enriquecer_e_analisar()

1. Carregando os dados brutos...
Encontrados 203 ASNs provedores únicos. Iniciando consultas à API...
-> Consultados 20/203 ASNs...
-> Consultados 40/203 ASNs...
-> Consultados 60/203 ASNs...
-> Consultados 80/203 ASNs...
-> Consultados 100/203 ASNs...
-> Consultados 120/203 ASNs...
-> Consultados 140/203 ASNs...
-> Consultados 160/203 ASNs...
-> Consultados 180/203 ASNs...
-> Consultados 200/203 ASNs...
Consultas concluídas! Mapeando os países no DataFrame...

Arquivos CSV salvos com sucesso no Google Drive!

MÉTRICAS QUANTITATIVAS DO PROJETO (BGP)

[A] TOP 5 ASes PROVEDORES (UPSTREAMS ABSOLUTOS):
 asn_upstream pais_upstream  frequencia
         9924            TW       84642
         4780            TW       77065
        17709            TW       58452
         6939  Desconhecido       28560
         3491            HK       27555

[B] TOP 5 PAÍSES QUE MAIS FORNECEM ROTAS:
pais_upstream  frequencia
           TW      290578
 Desconhecido       94580
           HK       37332
       

In [ ]:
import pandas as pd

ARQUIVO_ENTRADA = '/content/drive/MyDrive/Trabalho_BGP/upstreams_enriquecidos.csv'
ARQUIVO_SAIDA_INT = '/content/drive/MyDrive/Trabalho_BGP/upstreams_internacionais.csv'

def limpar_e_isolar_internacionais():
    print("Carregando a base de dados enriquecida...")
    df = pd.read_csv(ARQUIVO_ENTRADA)

    # Identificando alguns dos "Desconhecidos"
    correcoes_manuais = {
        6939: 'US',   # Hurricane Electric (EUA)
        9002: 'GB',   # RETN (Reino Unido)
        2914: 'JP',   # NTT Communications (Japão)
        6453: 'US',   # TATA Communications (EUA/Global)
        13335: 'US',  # Cloudflare (EUA)
        1299: 'SE',   # Arelion / Telia (Suécia)
        3356: 'US',   # Lumen / Level3 (EUA)
        174: 'US',    # Cogent (EUA)
        6762: 'IT',   # Telecom Italia Sparkle (Itália)
        3491: 'US',   # PCCW Global (EUA)
        21859: 'US',  # Zenlayer Inc (EUA)
        20473: 'US',  # Vultr (EUA)
        10310: 'US',  # Yahoo (EUA)
        53587: 'US',  # Azure Technology (EUA)
        204844: 'GB'  # NCSE Network (Reino Unido)
    }

    print("Aplicando correções geopolíticas nos provedores...")
    def corrigir_pais(row):
        if row['pais_upstream'] == 'Desconhecido' and row['asn_upstream'] in correcoes_manuais:
            return correcoes_manuais[row['asn_upstream']]
        return row['pais_upstream']

    df['pais_upstream'] = df.apply(corrigir_pais, axis=1)

    # Isolando apenas as rotas internacionais
    print("3. Removendo rotas domésticas (TW) para isolar a fronteira internacional...")
    df_internacional = df[df['pais_upstream'] != 'TW'].copy()

    df_internacional.to_csv(ARQUIVO_SAIDA_INT, index=False)
    print(f"Base internacional salva com sucesso em: {ARQUIVO_SAIDA_INT}")

    # Agrupamento e impressão das métricas atualizadas
    print("\n" + "="*50)
    print("NOVO CENÁRIO: APENAS TRÂNSITO INTERNACIONAL")
    print("="*50)

    top_paises_int = df_internacional.groupby('pais_upstream')['frequencia'].sum().reset_index()
    top_paises_int = top_paises_int.sort_values(by='frequencia', ascending=False)

    print("\n[B] TOP 5 PAÍSES QUE MAIS CONECTAM TAIWAN AO MUNDO:")
    print(top_paises_int.head(5).to_string(index=False))

    rotas_us = top_paises_int[top_paises_int['pais_upstream'] == 'US']['frequencia'].sum() if 'US' in top_paises_int['pais_upstream'].values else 0
    rotas_cn = top_paises_int[top_paises_int['pais_upstream'] == 'CN']['frequencia'].sum() if 'CN' in top_paises_int['pais_upstream'].values else 0

    print("\n[C] COMPARAÇÃO GEOPOLÍTICA DIRETA (EUA x CHINA):")
    print(f"-> Rotas dependentes de infraestrutura dos EUA: {rotas_us}")
    print(f"-> Rotas dependentes de infraestrutura da China: {rotas_cn}")
    print("="*50 + "\n")

limpar_e_isolar_internacionais()

1. Carregando a base de dados enriquecida...
2. Aplicando correções geopolíticas nos provedores...
3. Removendo rotas domésticas (TW) para isolar a fronteira internacional...
Base internacional salva com sucesso em: /content/drive/MyDrive/Trabalho_BGP/upstreams_internacionais.csv

NOVO CENÁRIO: APENAS TRÂNSITO INTERNACIONAL

[B] TOP 5 PAÍSES QUE MAIS CONECTAM TAIWAN AO MUNDO:
pais_upstream  frequencia
           US       51525
           HK       37332
           GB       18610
           JP       14406
 Desconhecido       10748

[C] COMPARAÇÃO GEOPOLÍTICA DIRETA (EUA x CHINA):
-> Rotas dependentes de infraestrutura dos EUA: 51525
-> Rotas dependentes de infraestrutura da China: 11



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuração de estilo acadêmico do Seaborn
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 12})

# Caminhos dos arquivos
ARQUIVO_ENTRADA = '/content/drive/MyDrive/Trabalho_BGP/upstreams_internacionais.csv'
DIR_SAIDA = '/content/drive/MyDrive/Trabalho_BGP/'

def gerar_graficos():
    print("Carregando base de dados internacional...")
    try:
        df = pd.read_csv(ARQUIVO_ENTRADA)
    except FileNotFoundError:
        print(f"Erro: Arquivo {ARQUIVO_ENTRADA} não encontrado.")
        return

    # GRÁFICO 1: Pizza (Dependência Internacional Top-5)
    print("Gerando Gráfico 1: Distribuição de Países...")
    paises_freq = df.groupby('pais_upstream')['frequencia'].sum().sort_values(ascending=False)

    # Remove o 'Desconhecido' da contagem principal para forçá-lo no grupo "Outros"
    if 'Desconhecido' in paises_freq:
        freq_desconhecido = paises_freq['Desconhecido']
        paises_freq = paises_freq.drop('Desconhecido')
    else:
        freq_desconhecido = 0

    # Pega os 5 maiores países conhecidos
    top5_paises = paises_freq.head(5)

    # Soma o resto (do 6º em diante + desconhecidos)
    freq_outros = paises_freq.iloc[5:].sum() + freq_desconhecido

    # Monta a série final para o gráfico
    dados_pizza = pd.concat([top5_paises, pd.Series({'Outros': freq_outros})])

    plt.figure(figsize=(8, 8))
    cores = sns.color_palette("pastel")[0:len(dados_pizza)]

    plt.pie(dados_pizza, labels=dados_pizza.index, autopct='%1.1f%%',
            startangle=140, colors=cores, wedgeprops={'edgecolor': 'black', 'linewidth': 1})
    plt.title('Distribuição de Rotas Internacionais para Taiwan por País', fontsize=14, pad=20, weight='bold')

    caminho_g1 = os.path.join(DIR_SAIDA, 'grafico_paises.png')
    plt.savefig(caminho_g1, dpi=300, bbox_inches='tight')
    plt.close()

    # GRÁFICO 2: Barras Horizontais (Top 10 Upstreams)
    print("Gerando Gráfico 2: Top 10 ASNs Intermediários...")

    # Agrupa por ASN e País
    asns_freq = df.groupby(['asn_upstream', 'pais_upstream'])['frequencia'].sum().reset_index()
    top10_asns = asns_freq.sort_values(by='frequencia', ascending=False).head(10)

    # Cria a coluna de rótulo formatado
    top10_asns['rotulo'] = 'AS' + top10_asns['asn_upstream'].astype(str) + ' (' + top10_asns['pais_upstream'] + ')'

    plt.figure(figsize=(10, 6))
    ax2 = sns.barplot(x='frequencia', y='rotulo', data=top10_asns, palette='Blues_r')

    plt.title('Top 10 Provedores de Trânsito Estrangeiros para Taiwan', fontsize=14, pad=15, weight='bold')
    plt.xlabel('Frequência de Rotas no BGP (Plano de Controle)', fontsize=12)
    plt.ylabel('Sistema Autônomo (País)', fontsize=12)

    # Adiciona os números exatos no final de cada barra
    for p in ax2.patches:
        ax2.annotate(f'{int(p.get_width()):,}',
                     (p.get_width(), p.get_y() + p.get_height() / 2.),
                     ha='left', va='center', xytext=(5, 0), textcoords='offset points', fontsize=10)

    plt.xlim(0, top10_asns['frequencia'].max() * 1.15)

    caminho_g2 = os.path.join(DIR_SAIDA, 'grafico_top_asns.png')
    plt.savefig(caminho_g2, dpi=300, bbox_inches='tight')
    plt.close()

    # GRÁFICO 3: Comparação Geopolítica (EUA x China)
    print("Gerando Gráfico 3: Comparação EUA vs China...")

    # Filtra apenas US e CN
    df_us_cn = df[df['pais_upstream'].isin(['US', 'CN'])].copy()
    comp_freq = df_us_cn.groupby('pais_upstream')['frequencia'].sum().reindex(['US', 'CN'], fill_value=0).reset_index()

    # Mapeia nomes mais amigáveis para o gráfico
    mapeamento_nomes = {'US': 'Estados Unidos (US)', 'CN': 'China Continental (CN)'}
    comp_freq['pais_nome'] = comp_freq['pais_upstream'].map(mapeamento_nomes)

    plt.figure(figsize=(7, 6))
    ax3 = sns.barplot(x='pais_nome', y='frequencia', data=comp_freq, palette=['#1f77b4', '#d62728'])

    plt.title('Dependência de Rotas BGP: Estados Unidos vs China', fontsize=14, pad=15, weight='bold')
    plt.xlabel('Jurisdição da Infraestrutura Intermediária', fontsize=12)
    plt.ylabel('Frequência Total de Rotas', fontsize=12)

    # Adiciona os rótulos de dados em cima das barras
    for p in ax3.patches:
        ax3.annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', fontsize=12, weight='bold')

    # Expande um pouco o eixo Y
    plt.ylim(0, comp_freq['frequencia'].max() * 1.15)

    caminho_g3 = os.path.join(DIR_SAIDA, 'grafico_us_cn.png')
    plt.savefig(caminho_g3, dpi=300, bbox_inches='tight')
    plt.close()

    print("\nSucesso! Todos os gráficos foram gerados e salvos com sucesso na pasta do projeto.")
    print("- grafico_paises.png")
    print("- grafico_top_asns.png")
    print("- grafico_us_cn.png")

gerar_graficos()

Carregando base de dados internacional...
Gerando Gráfico 1: Distribuição de Países...
Gerando Gráfico 2: Top 10 ASNs Intermediários...


/tmp/ipykernel_1082/3960758422.py:70: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  ax2 = sns.barplot(x='frequencia', y='rotulo', data=top10_asns, palette='Blues_r')


Gerando Gráfico 3: Comparação EUA vs China...


/tmp/ipykernel_1082/3960758422.py:103: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax3 = sns.barplot(x='pais_nome', y='frequencia', data=comp_freq, palette=['#1f77b4', '#d62728'])



Sucesso! Todos os gráficos foram gerados e salvos com sucesso na pasta do projeto.
- grafico_paises.png
- grafico_top_asns.png
- grafico_us_cn.png


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12})

ARQUIVO_ENTRADA = '/content/drive/MyDrive/Trabalho_BGP/upstreams_internacionais.csv'
DIR_SAIDA = '/content/drive/MyDrive/Trabalho_BGP/'

def gerar_grafico_top10_cn():
    print("Carregando base de dados internacional...")
    try:
        df = pd.read_csv(ARQUIVO_ENTRADA)
    except FileNotFoundError:
        print(f"Erro: Arquivo {ARQUIVO_ENTRADA} não encontrado.")
        return

    # Agrupa por país e ordena
    paises_freq = df.groupby('pais_upstream')['frequencia'].sum().reset_index()
    paises_freq = paises_freq.sort_values(by='frequencia', ascending=False)

    # Extrai os Top 10 países
    top_10_siglas = paises_freq.head(10)['pais_upstream'].tolist()

    # Garante que a China (CN) esteja na lista final, mesmo que tenha frequência baixa
    if 'CN' not in top_10_siglas:
        top_10_siglas.append('CN')

    # Filtra o DataFrame apenas para esses países selecionados
    df_plot = paises_freq[paises_freq['pais_upstream'].isin(top_10_siglas)].copy()

    # Dicionário para traduzir as siglas no eixo X
    mapa_nomes = {
        'US': 'Estados Unidos',
        'HK': 'Hong Kong',
        'GB': 'Reino Unido',
        'JP': 'Japão',
        'SG': 'Singapura',
        'MY': 'Malásia',
        'KR': 'Coreia do Sul',
        'AU': 'Austrália',
        'PH': 'Filipinas',
        'ID': 'Indonésia',
        'CN': 'China',
        'Desconhecido': 'Desconhecido'
    }

    df_plot['pais_nome'] = df_plot['pais_upstream'].map(lambda x: mapa_nomes.get(x, x))

    # Ordena o DataFrame seguindo estritamente a ordem da nossa lista (Top 10 + CN)
    df_plot['pais_upstream'] = pd.Categorical(df_plot['pais_upstream'], categories=top_10_siglas, ordered=True)
    df_plot = df_plot.sort_values('pais_upstream')

    cores = []
    for pais in df_plot['pais_upstream']:
        if pais == 'US':
            cores.append('#1f77b4')
        elif pais == 'CN':
            cores.append('#d62728')
        else:
            cores.append('#b0bec5')

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x='pais_nome', y='frequencia', data=df_plot, palette=cores)

    plt.title('Dependência de Rotas Internacionais BGP: Top 10 Países + China', fontsize=15, pad=20, weight='bold')
    plt.xlabel('Jurisdição da Infraestrutura Intermediária', fontsize=12, labelpad=10)
    plt.ylabel('Frequência Total de Rotas', fontsize=12, labelpad=10)
    plt.xticks(rotation=45, ha='right')

    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', fontsize=10)

    plt.ylim(0, df_plot['frequencia'].max() * 1.15)

    caminho_saida = os.path.join(DIR_SAIDA, 'grafico_comparativo_amplo.png')
    plt.savefig(caminho_saida, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"Sucesso! Gráfico atualizado salvo como: {caminho_saida}")

gerar_grafico_top10_cn()

Carregando base de dados internacional...


/tmp/ipykernel_1082/1788693672.py:68: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x='pais_nome', y='frequencia', data=df_plot, palette=cores)


Sucesso! Gráfico atualizado salvo como: /content/drive/MyDrive/Trabalho_BGP/grafico_comparativo_amplo.png
